# **Preprocessing Text — Tiền xử lý Văn bản**

## Mục lục
- [1. Cài đặt & Khai báo Thư viện](#1-cài-đặt--khai-báo-thư-viện)
- [2. Tải và đọc Dữ liệu](#2-tải-và-đọc-dữ-liệu)
- [3. Hàm tiền xử lý dùng chung](#3-hàm-tiền-xử-lý-dùng-chung)
- [4. Các bước tiền xử lý riêng lẻ](#4-các-bước-tiền-xử-lý-riêng-lẻ)
- [5. Pipeline chuẩn hóa văn bản và phân tích định lượng](#5-pipeline-chuẩn-hóa-văn-bản-và-phân-tích-định-lượng)
- [6. So sánh chiến lược Tokenization](#6-so-sánh-chiến-lược-tokenization)
- [7. Loại bỏ Stop Words và phân tích thông tin](#7-loại-bỏ-stop-words-và-phân-tích-thông-tin)
- [8. Stemming, Lemmatization và so sánh định lượng](#8-stemming-lemmatization-và-so-sánh-định-lượng)
- [9. Vector hóa văn bản và phân tích không gian đặc trưng](#9-vector-hóa-văn-bản-và-phân-tích-không-gian-đặc-trưng)

## 1. Cài đặt & Khai báo **Thư viện** và **Gói phụ thuộc**

In [ ]:
import os
import re
import time
import warnings
import tracemalloc
import glob
import random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_notebook

# Scientific/Statistical libraries
from scipy import sparse, fftpack, stats
from scipy.ndimage import gaussian_filter1d
from scipy.stats import linregress
from IPython.display import display, HTML

# NLP libraries
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

# Gensim
import gensim
from gensim.models import Word2Vec

# Tokenizers
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

# Sentence transformers
from sentence_transformers import SentenceTransformer

# Scikit-learn - Feature extraction and preprocessing
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.feature_selection import mutual_info_classif

# Scikit-learn - Model selection and validation
from sklearn.model_selection import cross_val_score, train_test_split

# Scikit-learn - Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Scikit-learn - Clustering
from sklearn.cluster import KMeans

# Scikit-learn - Dimensionality reduction
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD, PCA

# Scikit-learn - Metrics
from sklearn.metrics import (
    accuracy_score, f1_score, silhouette_score,
    classification_report, precision_recall_fscore_support,
    roc_curve, auc
)
from sklearn.metrics.pairwise import cosine_similarity

# Scikit-learn - Pipeline
from sklearn.pipeline import Pipeline

# Package installation (usually done once)
# %pip install spacy
# %pip install gensim

# --- Load NLTK resources ---
nltk.download('punkt',    quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',  quiet=True)
nltk.download('omw-1.4',  quiet=True)

# --- spaCy pipeline ---
nlp_spacy = spacy.blank('en')


## 2. Tải và đọc Dữ liệu

In [ ]:
df = pd.read_csv('../data\\raw/IMDB Dataset.csv')
print(df.shape)
df.head()


## 3. Hàm tiền xử lý dùng chung

Toàn bộ pipeline tiền xử lý trong notebook này đều dựa trên một hàm chuẩn hóa duy nhất — `basic_clean()`. Hàm này thực thi tuần tự các bước làm sạch cơ bản, bao gồm: chuyển về chữ thường, loại bỏ thẻ HTML, URL, mention, hashtag và chuẩn hóa khoảng trắng.

In [ ]:
def basic_clean(text):
    """Pipeline chuẩn hóa cơ bản dùng chung cho toàn notebook."""
    text = str(text).lower()
    text = re.sub(r'<.*?>',            ' ', text)   # Remove HTML
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # Remove URL
    text = re.sub(r'(?<!\w)@\w+',    ' ', text)   # Remove mention
    text = re.sub(r'(?<!\w)#\w+',    ' ', text)   # Remove hashtag
    text = re.sub(r'\s+',             ' ', text).strip()  # Normalize whitespace
    return text
                            

## 4. Các bước tiền xử lý riêng lẻ

Mỗi bước trong pipeline `basic_clean()` được tách thành một hàm độc lập dưới đây, phục vụ mục đích tham khảo hoặc áp dụng lẻ khi cần kiểm soát chi tiết từng bước xử lý.

### 4.1 Lowercase — Chuyển chữ thường

In [ ]:
def to_lowercase(text): 
    return str(text).lower()
test = f"My name is Pham Quoc Khanh"
test = to_lowercase(test)
print(test)
# Kết quả: my name is pham quoc khanh


### 4.2 Remove HTML Tags — Loại bỏ thẻ HTML

Dữ liệu IMDB chứa nhiều thẻ HTML thừa (đặc biệt là `<br />`) từ quá trình crawling. Biểu thức chính quy `<.*?>` sử dụng **lazy matching** để khớp chính xác từng thẻ, tránh xóa nhầm nội dung giữa các thẻ.

In [ ]:
def remove_html(text):
    pattern =  re.compile('<.*?>')
    return pattern.sub(r' ', text)
test_str = "<p>Hello <b>world</b>!</p>"
result = remove_html(test_str)
print(result)  
# Kết quả: Hello world!


### 4.3 Remove URL — Loại bỏ đường dẫn

Các URL dạng `http://`, `https://` hoặc `www.` không mang thông tin cảm xúc, cần được loại bỏ trước khi phân tích từ vựng.

In [ ]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

test_str = "Visit https://github.com or http://example.com for more info. Or check www.example.org!"
result = remove_url(test_str)
print(result) 
# Kết quả: Visit  or  for more info. Or check 


### 4.4 Remove Mention — Loại bỏ mention người dùng

Chuỗi `@username` không xuất hiện phổ biến trong IMDB nhưng vẫn được xử lý để đảm bảo tính tổng quát của pipeline.

In [ ]:
def remove_mention(text):
    pattern = re.compile(r'(?<!\w)@\w+')
    return pattern.sub(r'', text)

test_str = "Hello @user1, please meet @user2 and @user_3!"
result = remove_mention(test_str)
print(result)  
# Kết quả: Hello , please meet  and !


### 4.5 Remove Hashtag — Loại bỏ hashtag

Tương tự mention, hashtag (`#topic`) là nhiễu không liên quan đến nội dung đánh giá phim.

In [ ]:
def remove_hashtag(text):
    pattern = re.compile(r'(?<!\w)#\w+')
    return pattern.sub(r'', text)

test_str = "This is a #test string with #hashtags and #123abc."
result = remove_hashtag(test_str)
print(result)
# Kết quả: This is a  string with  and .


### 4.6 Remove Special Characters — Loại bỏ ký tự đặc biệt

Loại bỏ toàn bộ ký tự ngoài chữ cái, chữ số và khoảng trắng. Tham số `keep_apostrophe=True` giữ lại dấu nháy đơn để bảo toàn các dạng viết tắt (`don't`, `it's`), tránh làm hỏng ngữ nghĩa của từ phủ định.

In [ ]:
def remove_special_characters(text, keep_apostrophe=True):
    # Keep letters/spaces; optionally keep apostrophes in words like don't.
    if keep_apostrophe:
        pattern = re.compile(r"[^\w\s']")
    else:
        pattern = re.compile(r'[^\w\s]')
    return pattern.sub('', text)

text = "Hello! It's a test: don't remove apostrophes, ok?"
print(remove_special_characters(text, keep_apostrophe=True))
# Kết quả: Hello It's a test don't remove apostrophes ok

print(remove_special_characters(text, keep_apostrophe=False))
# Kết quả: Hello Its a test dont remove apostrophes ok


### 4.7 Remove Numbers — Loại bỏ số

Hai chế độ xử lý được hỗ trợ:
* **`standalone`**: Chỉ xóa các token thuần số (VD: `2024`, `100`), giữ lại các dạng như `50s` hay `MP3`.
* **`all`**: Xóa toàn bộ ký tự số, kể cả khi nằm trong từ (VD: `movie2` → `movie`).

In [ ]:
def remove_numbers(text, mode='standalone'):
    # mode='standalone': remove tokens that are only digits (e.g., 2024, 10/10 -> 10 parts).
    # mode='all': remove every digit character, including inside words (movie2 -> movie).
    if mode == 'all':
        pattern = re.compile(r'\d+')
    else:
        pattern = re.compile(r'\b\d+\b')
    return pattern.sub('', text)    

# Change mode to 'all' if you want to remove every digit character.
text = "The year is 2024. I watched 10/10 movies. This is movie2."
print(remove_numbers(text, mode='standalone'))
# Kết quả: The year is . I watched / movies. This is movie2.

print(remove_numbers(text, mode='all'))
# Kết quả: The year is . I watched / movies. This is movie.


### 4.8 Normalize Whitespace — Chuẩn hóa khoảng trắng

Bước cuối trong pipeline: gộp nhiều khoảng trắng liên tiếp (do các bước xóa trước tạo ra) thành một khoảng trắng duy nhất và cắt hai đầu chuỗi.

In [ ]:
def normalize_whitespace(text):
    # Replace repeated spaces/newlines/tabs with a single space, then trim.
    return re.sub(r'\s+', ' ', text).strip()

text = "This   is\ta   test.\nNew   line.\n\n"
print(normalize_whitespace(text))
# Kết quả: This is a test. New line.


## 5. Pipeline chuẩn hóa văn bản và phân tích định lượng

### Mục tiêu
Thay vì chỉ áp dụng từng bước tiền xử lý một cách mù quáng, thực nghiệm này đo lường **tác động định lượng** của mỗi bước lên hai chiều kích thước:
1. **Không gian từ vựng**: `vocab_change_ratio` — tỉ lệ từ bị thay đổi hoặc loại bỏ so với bước trước.
2. **Phân phối độ dài văn bản**: mean, median và p90 theo số từ và số ký tự.

### Thiết kế thực nghiệm
* Dữ liệu được xử lý tuần tự qua 8 bước, mỗi bước nhận đầu ra của bước trước làm đầu vào.
* Với mỗi bước, ghi nhận `prev_vocab_size`, `curr_vocab_size` và `vocab_change_ratio` để theo dõi sự co hẹp của không gian đặc trưng.
* Kết quả được tổng hợp thành một `DataFrame` duy nhất để so sánh trực quan.

In [ ]:
# Khởi tạo pipeline
class TextCleaningPipeline:
    def __init__(self):
        self.steps = []

    def add_step(self, name, func):
        self.steps.append((name, func))
        return self

    def _tokenize(self, series):
        return series.str.split()

    def _vocab_set(self, series):
        tokens = self._tokenize(series).explode().dropna()
        return set(tokens.tolist())

    def _length_stats(self, series):
        word_len = self._tokenize(series).str.len()
        char_len = series.str.len()
        return {
            'doc_count': int(series.shape[0]),
            'word_len_mean': float(word_len.mean()),
            'word_len_median': float(word_len.median()),
            'word_len_p90': float(word_len.quantile(0.9)),
            'char_len_mean': float(char_len.mean()),
            'char_len_median': float(char_len.median()),
            'char_len_p90': float(char_len.quantile(0.9)),
        }

    def _vocab_change_ratio(self, prev_vocab, curr_vocab):
        if len(prev_vocab) == 0:
            return {'prev_vocab_size': 0, 'curr_vocab_size': len(curr_vocab), 'vocab_change_ratio': 0.0}
        overlap = len(prev_vocab & curr_vocab)
        changed_ratio = 1 - (overlap / len(prev_vocab))
        return {
            'prev_vocab_size': len(prev_vocab),
            'curr_vocab_size': len(curr_vocab),
            'vocab_change_ratio': float(changed_ratio),
        }

    def run_and_report(self, series):
        current_series = series.copy().astype(str)
        reports = []

        # Đánh giá dữ liệu Raw
        prev_vocab = self._vocab_set(current_series)
        stats = self._length_stats(current_series)
        
        reports.append({
            'step': 'raw',
            'prev_vocab_size': np.nan,
            'curr_vocab_size': len(prev_vocab),
            'vocab_change_ratio': np.nan,
            **stats
        })

        # Chạy từng bước và đánh giá
        for step_name, func in self.steps:
            current_series = current_series.apply(func)
            curr_vocab = self._vocab_set(current_series)
            vocab_metrics = self._vocab_change_ratio(prev_vocab, curr_vocab)
            stats = self._length_stats(current_series)
            
            reports.append({
                'step': step_name,
                **vocab_metrics,
                **stats
            })
            prev_vocab = curr_vocab 

        # Format dataframe
        report_df = pd.DataFrame(reports)
        report_df['vocab_change_ratio'] = report_df['vocab_change_ratio'].round(4)
        float_cols = ['word_len_mean', 'word_len_median', 'word_len_p90', 
                      'char_len_mean', 'char_len_median', 'char_len_p90']
        for col in float_cols:
            report_df[col] = report_df[col].round(2)

        return current_series, report_df

# Thực thi
df_eval = df.copy()
df_eval['review_raw'] = df_eval['review'].astype(str)

pipeline = TextCleaningPipeline()
pipeline.add_step("lowercase", to_lowercase) \
        .add_step("remove_html", remove_html) \
        .add_step("remove_url", remove_url) \
        .add_step("remove_mention", remove_mention) \
        .add_step("remove_hashtag", remove_hashtag) \
        .add_step("remove_special_characters", remove_special_characters) \
        .add_step("remove_numbers_standalone", remove_numbers) \
        .add_step("normalize_whitespace", normalize_whitespace)

# Chạy pipeline và cập nhật cột text đã làm sạch vào df_eval
df_eval['review_clean'], report_df = pipeline.run_and_report(df_eval['review_raw'])

report_df


### Phân tích tác động lên phân phối độ dài theo từng bước

Bảng dưới đây bổ sung các cột delta (chênh lệch giữa hai bước liên tiếp) nhằm xác định bước nào làm thay đổi mạnh nhất phần thân phân phối (mean) và phần đuôi phân phối (p90).

In [ ]:
impact_df = report_df.copy()
impact_df['word_len_mean_delta'] = impact_df['word_len_mean'].diff().round(2)
impact_df['word_len_p90_delta'] = impact_df['word_len_p90'].diff().round(2)
impact_df['char_len_mean_delta'] = impact_df['char_len_mean'].diff().round(2)
impact_df['char_len_p90_delta'] = impact_df['char_len_p90'].diff().round(2)

impact_df[[
    'step',
    'vocab_change_ratio',
    'word_len_mean', 'word_len_mean_delta',
    'word_len_p90', 'word_len_p90_delta',
    'char_len_mean', 'char_len_mean_delta',
    'char_len_p90', 'char_len_p90_delta'
]]


### Phân tích kết quả

**1. Các bước có tác động lớn đến không gian từ vựng**

* **`lowercase`**: Bước đầu tiên và có tác động rất rộng về `vocab_change_ratio` (0.4418). Hàng nghìn từ như `The`, `THE`, `the` được gộp về cùng một type — giảm mạnh kích thước từ điển mà không làm mất thông tin.
* **`remove_html`**: Vocab thay đổi đáng kể (một số token bị dính thẻ được tách ra) do các thẻ `<br />`, `<i>`, `<b>` chiếm một lượng lớn trong dữ liệu thô IMDB.
* **`remove_special_characters`**: Có **tác động lớn nhất** đến `vocab_change_ratio` (0.7519) và `char_len_mean` vì loại bỏ toàn bộ dấu câu, ký hiệu và các chuỗi nhiễu, giúp làm sạch sâu không gian đặc trưng.

**2. Các bước có tác động không đáng kể**

* **`remove_url`** và **`remove_mention`**: `vocab_change_ratio` gần bằng 0. Dữ liệu IMDB không chứa nhiều URL hay mention — xác nhận tính đặc thù của nguồn dữ liệu.
* **`remove_hashtag`**: Tương tự, hashtag vắng mặt trong reviews điện ảnh.

**3. Xu hướng phân phối độ dài**

* `word_len_mean` và `char_len_mean` giảm dần qua từng bước — đây là hành vi kỳ vọng đúng của pipeline làm sạch.
* `word_len_p90` giảm nhẹ, trong khi `char_len_mean` giảm mạnh nhất sau `remove_special_characters` (~35 ký tự), cho thấy phần lớn "độ dài ảo" đến từ ký tự nhiễu chứ không phải nội dung thực.

**4. Định hướng**

Sau 8 bước, văn bản đã được chuẩn hóa hoàn toàn. Đầu ra này đủ điều kiện làm đầu vào cho các bước phân tích tiếp theo: so sánh chiến lược Tokenization (Section 6) và đánh giá tác động của Stop Words (Section 7).


## 6. So sánh chiến lược Tokenization

### Mục tiêu
So sánh bốn nhóm chiến lược tokenization đại diện — word-level, sentence-level, character-level và subword (BPE) — trên cùng một tập dữ liệu đã qua làm sạch cơ bản. Với word-level, thực nghiệm cả hai tokenizer: NLTK và spaCy.

### Cơ sở lý thuyết

#### 1. Word-level Tokenization
Phân tách văn bản thành các từ dựa trên khoảng trắng và dấu câu. Đây là chiến lược phổ biến nhất trong NLP truyền thống.
* **NLTK `word_tokenize`**: Sử dụng mô hình Punkt, xử lý tốt dấu câu và viết tắt.
* **spaCy**: Dựa trên tập quy tắc ngôn ngữ học, nhanh hơn và nhất quán hơn với văn bản thực tế.

#### 2. Sentence-level Tokenization
Phân tách thành các câu thay vì từ. Phù hợp cho các mô hình cần ngữ cảnh câu như BERT hay các bài toán tóm tắt.

#### 3. Character-level Tokenization
Mỗi ký tự là một token. Từ vựng cực nhỏ (~100 types) nhưng chuỗi token rất dài, phù hợp để xử lý lỗi chính tả và ngôn ngữ ít tài nguyên.

#### 4. Subword BPE (Byte Pair Encoding)
Thuật toán học cách phân tách từ từ dữ liệu thực tế. BPE cân bằng giữa word-level và character-level: giữ nguyên các từ phổ biến, phân tách các từ hiếm thành các subword có nghĩa.

### Chỉ số đánh giá
* **`vocab_size`**: Kích thước từ vựng trên tập train — phản ánh độ phức tạp của không gian đặc trưng.
* **`oov_ratio_test`**: Tỉ lệ token ở tập test không xuất hiện trong từ vựng train — đo lường mức độ tổng quát hóa.
* **`avg_token_seq_len_test`**: Độ dài chuỗi token trung bình — ảnh hưởng trực tiếp đến chi phí tính toán.

### Cài đặt
Dữ liệu đầu vào được làm sạch nhẹ bằng `basic_clean()` để đảm bảo sự công bằng giữa các chiến lược. Tập dữ liệu được chia 80/20 (train/test) với `random_state=42` để đảm bảo tính tái lập.

In [ ]:
df_tok = df[['review', 'sentiment']].copy()
df_tok['text_clean'] = df_tok['review'].apply(basic_clean)

train_texts, test_texts = train_test_split(
    df_tok['text_clean'],
    test_size=0.2,
    random_state=42,
    shuffle=True,
 )

train_text_list = train_texts.tolist()
test_text_list = test_texts.tolist()

def flatten_token_lists(token_lists):
    return [tok for doc in token_lists for tok in doc]

def oov_ratio(test_token_lists, train_vocab):
    test_tokens = flatten_token_lists(test_token_lists)
    if len(test_tokens) == 0:
        return 0.0
    oov_count = sum(tok not in train_vocab for tok in test_tokens)
    return oov_count / len(test_tokens)

def evaluate_strategy(name, train_token_lists, test_token_lists):
    train_vocab = set(flatten_token_lists(train_token_lists))
    avg_len_test = float(np.mean([len(x) for x in test_token_lists]))
    return {
        'strategy': name,
        'vocab_size': int(len(train_vocab)),
        'oov_ratio_test': float(oov_ratio(test_token_lists, train_vocab)),
        'avg_token_seq_len_test': avg_len_test,
    }

def spacy_word_tokenize(text):
    return [tok.text for tok in nlp_spacy.make_doc(text)]

strategy_tokenizers = {
    'word_level_nltk': word_tokenize,
    'word_level_spacy': spacy_word_tokenize,
    'sentence_level': sent_tokenize,
    'character_level': list,
}

comparison_rows = []
for strategy_name, tokenizer_fn in strategy_tokenizers.items():
    train_tokens = [tokenizer_fn(text) for text in train_text_list]
    test_tokens = [tokenizer_fn(text) for text in test_text_list]
    comparison_rows.append(evaluate_strategy(strategy_name, train_tokens, test_tokens))

bpe_tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))
bpe_tokenizer.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
bpe_trainer = trainers.BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]'],
)
bpe_tokenizer.train_from_iterator(train_text_list, trainer=bpe_trainer)

train_bpe = [bpe_tokenizer.encode(text).tokens for text in train_text_list]
test_bpe = [bpe_tokenizer.encode(text).tokens for text in test_text_list]
comparison_rows.append(evaluate_strategy('subword_bpe_hf_tokenizers', train_bpe, test_bpe))

tokenization_comparison_df = pd.DataFrame(comparison_rows)
tokenization_comparison_df['oov_ratio_test'] = tokenization_comparison_df['oov_ratio_test'].round(6)
tokenization_comparison_df['avg_token_seq_len_test'] = tokenization_comparison_df['avg_token_seq_len_test'].round(2)

tokenization_comparison_df  


### Phân tích kết quả

**1. Kích thước từ vựng (`vocab_size`)**

* **Character-level** có `vocab_size` nhỏ nhất (158 types — toàn bộ bảng ký tự ASCII). Đây là không gian đặc trưng tối giản nhất nhưng không mang tính ngữ nghĩa.
* **Sentence-level** tạo ra từ vựng **lớn nhất** (hơn 470,000 types) do các câu gần như duy nhất, phản ánh tính đa dạng ở cấp độ cấu trúc câu.
* **Subword BPE** (15,883 vocab) cân bằng giữa hai cực trên — từ vựng có giới hạn nhưng vẫn nắm bắt được các morpheme có nghĩa.

**2. Tỉ lệ OOV (`oov_ratio_test`)**

* **Character-level**: OOV thực tế bằng 0 — không thể có ký tự "lạ" vì bảng chữ cái là cố định.
* **Sentence-level**: OOV **cao nhất** (~0.955) vì tập test chứa hầu hết các câu mới không có trong tập train.
* **Subword BPE**: OOV rất thấp (0.000008) vì từ lạ được phân tách thành các subword đã biết, thay vì map sang `[UNK]` như word-level.
* **Word-level (NLTK/spaCy)**: OOV ở mức trung bình (~0.005-0.007) — đây là điểm yếu cổ điển của chiến lược này khi gặp từ mới.

**3. Độ dài chuỗi token (`avg_token_seq_len_test`)**

* **Character-level** tạo ra chuỗi dài nhất (~1291 tokens) — chi phí tính toán đắt đỏ.
* **Word-level** và **Subword BPE** có độ dài chuỗi tương đương (~265-288 tokens) — phù hợp với các mô hình Transformer.

**4. Định hướng lựa chọn chiến lược**

Với bài toán Sentiment Analysis trên IMDB, **word-level tokenization (NLTK hoặc spaCy)** là lựa chọn tiêu chuẩn cho các mô hình truyền thống (BoW, TF-IDF). Với các mô hình Transformer (BERT, RoBERTa), **Subword BPE** là lựa chọn mặc định.


## 7. Loại bỏ Stop Words và phân tích thông tin

### Mục tiêu
Đánh giá tác động của việc loại bỏ stop words lên ba tiêu chí định lượng:
1. **Kích thước từ vựng** trước và sau loại bỏ — phản ánh mức độ nén không gian đặc trưng.
2. **Mutual Information (MI) trung bình** giữa từ và nhãn — đo lường lượng thông tin phân loại được giữ lại.
3. **Hiệu năng Naive Bayes** (Accuracy và F1-macro) — đánh giá tác động thực tế lên mô hình.

### Cơ sở lý thuyết

#### 1. Stop Words là gì?
Stop words (từ dừng) là các từ chức năng xuất hiện với tần suất rất cao trong mọi văn bản (`the`, `a`, `is`, `in`...) nhưng không mang thông tin phân biệt nhãn. Theo phân phối Zipf, chúng chiếm phần lớn tổng số token nhưng đóng góp rất ít vào Information Gain.

#### 2. Tại sao MI trung bình lại tăng sau khi loại bỏ stop words?
Mutual Information giữa từ $w$ và nhãn $y$ được tính bằng:
$$MI(w; y) = \sum_{w, y} P(w, y) \log \frac{P(w, y)}{P(w)P(y)}$$
Các stop words xuất hiện đồng đều ở cả lớp Positive lẫn Negative, khiến $P(w | \text{pos}) \approx P(w | \text{neg}) \approx P(w)$, dẫn đến $MI \approx 0$. Khi loại bỏ chúng, MI trung bình của không gian đặc trưng còn lại sẽ tăng lên.

#### 3. Giới hạn của danh sách stop words mặc định NLTK
Danh sách stop words của NLTK bao gồm các từ phủ định như `not`, `no`, `nor`. Trong bài toán phân tích cảm xúc, đây là những từ mang tính phân biệt quan trọng (VD: *"not good"* vs *"good"*). Loại bỏ chúng có thể gây mất ngữ nghĩa và làm giảm hiệu năng mô hình.

### Thiết kế thực nghiệm
* Dữ liệu được chia 80/20 với stratified sampling để đảm bảo tỉ lệ nhãn cân bằng giữa train và test.
* `CountVectorizer` với `min_df=2` được sử dụng để loại bỏ từ hiếm không liên quan.
* Mô hình Naive Bayes được chọn vì phù hợp với đặc trưng thưa và cho phép so sánh nhanh.

In [ ]:
df_stop = df[['review', 'sentiment']].copy()
df_stop['text_clean'] = df_stop['review'].apply(basic_clean)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_stop['text_clean'],
    df_stop['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=df_stop['sentiment']
 )

stop_words_en = set(stopwords.words('english'))
y_train_bin = (y_train == 'positive').astype(int).to_numpy()

def evaluate_stopword_setting(remove_stopwords):
    vectorizer = CountVectorizer(
        stop_words=sorted(stop_words_en) if remove_stopwords else None,
        token_pattern=r'(?u)\b\w+\b',
        min_df=2
    )

    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    vocab_size = len(vectorizer.vocabulary_)

    mi_scores = mutual_info_classif(
        X_train,
        y_train_bin,
        discrete_features=True,
        random_state=42
    )
    mean_mi = float(np.mean(mi_scores))

    clf = MultinomialNB()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    return {
        'vocab_size': int(vocab_size),
        'mean_mi': float(mean_mi),
        'nb_accuracy': float(accuracy_score(y_test, y_pred)),
        'nb_f1_macro': float(f1_score(y_test, y_pred, average='macro'))
    }

before_stats = evaluate_stopword_setting(remove_stopwords=False)
after_stats = evaluate_stopword_setting(remove_stopwords=True)

stopword_comparison_df = pd.DataFrame([
    {
        'metric': 'vocab_size',
        'before': before_stats['vocab_size'],
        'after': after_stats['vocab_size'],
        'delta_after_minus_before': after_stats['vocab_size'] - before_stats['vocab_size']
    },
    {
        'metric': 'mean_mi',
        'before': before_stats['mean_mi'],
        'after': after_stats['mean_mi'],
        'delta_after_minus_before': after_stats['mean_mi'] - before_stats['mean_mi']
    },
    {
        'metric': 'nb_accuracy',
        'before': before_stats['nb_accuracy'],
        'after': after_stats['nb_accuracy'],
        'delta_after_minus_before': after_stats['nb_accuracy'] - before_stats['nb_accuracy']
    },
    {
        'metric': 'nb_f1_macro',
        'before': before_stats['nb_f1_macro'],
        'after': after_stats['nb_f1_macro'],
        'delta_after_minus_before': after_stats['nb_f1_macro'] - before_stats['nb_f1_macro']
    }
])

stopword_comparison_df


### Phân tích kết quả — Thảo luận

Phần dưới tính toán và hiển thị sự thay đổi của từng chỉ số sau khi áp dụng loại bỏ stop words, đồng thời đưa ra nhận xét về tính hiệu quả của thao tác này trong bài toán phân tích cảm xúc.

In [ ]:
acc_delta = after_stats['nb_accuracy'] - before_stats['nb_accuracy']
f1_delta  = after_stats['nb_f1_macro'] - before_stats['nb_f1_macro']
mi_delta  = after_stats['mean_mi']     - before_stats['mean_mi']

summary_df = pd.DataFrame({
    'Chỉ số': ['vocab_size', 'mean_mi', 'nb_accuracy', 'nb_f1_macro'],
    'Trước khi loại bỏ': [
        before_stats['vocab_size'],
        round(before_stats['mean_mi'], 6),
        round(before_stats['nb_accuracy'], 4),
        round(before_stats['nb_f1_macro'], 4),
    ],
    'Sau khi loại bỏ': [
        after_stats['vocab_size'],
        round(after_stats['mean_mi'], 6),
        round(after_stats['nb_accuracy'], 4),
        round(after_stats['nb_f1_macro'], 4),
    ],
    'Delta': [
        after_stats['vocab_size'] - before_stats['vocab_size'],
        round(mi_delta, 6),
        round(acc_delta, 4),
        round(f1_delta, 4),
    ]
})

summary_df


### Phân tích kết quả

**1. Tác động lên không gian từ vựng**

* Kích thước từ vựng (`vocab_size`) giảm nhẹ sau khi loại bỏ stop words (giảm 152 types). Việc loại bỏ các từ dừng không làm giảm mạnh kích thước từ vựng nhưng giúp tập trung vào các từ mang ngữ nghĩa.

**2. Tác động lên chất lượng đặc trưng (Mutual Information)**

* `mean_mi` **giảm nhẹ** (từ 0.000088 xuống 0.000083) sau khi loại bỏ stop words. Mặc dù lý thuyết kỳ vọng MI tăng khi bỏ các từ gây nhiễu, nhưng thực tế với danh sách mặc định NLTK, việc loại bỏ cả các từ quan trọng (như từ phủ định) có thể làm giảm lượng thông tin phân loại chung.

**3. Tác động lên hiệu năng mô hình**

* Accuracy và F1-macro của Naive Bayes **tăng nhẹ** (delta ~0.018). Điều này cho thấy việc loại bỏ stop words có lợi cho mô hình xác suất đơn giản, giúp giảm nhiễu từ các hư từ có tần suất cao.

**4. Lưu ý về từ phủ định**

* Danh sách NLTK bao gồm `not`, `no`, `nor` — những từ mang tính phân cực quan trọng. Trong thực tế, nên sử dụng danh sách stop words tùy chỉnh để tránh mất ngữ nghĩa phủ định.


## 8. Stemming, Lemmatization và so sánh định lượng

### Cơ sở lý thuyết

#### 1. Stemming
Stemming là kỹ thuật loại bỏ các hậu tố (đôi khi là tiền tố) của từ để đưa từ về dạng gốc (stem). Quá trình này hoàn toàn dựa trên quy tắc chuỗi ký tự (heuristics), không quan tâm đến ngữ cảnh hay ngữ pháp. Do đó, stem sinh ra có thể không phải là một từ có nghĩa trong từ điển.

* **Porter Stemmer:** Thuật toán lâu đời và phổ biến nhất cho tiếng Anh. Áp dụng tuần tự các quy tắc cắt đuôi (`-ing`, `-ly`, `-ed`). Tốc độ nhanh nhưng đôi khi cắt quá mức (VD: `generously` → `generous` → `generos`).
* **Snowball Stemmer:** Cải tiến của Porter, quyết liệt hơn trong việc rút gọn từ vựng. Hỗ trợ đa ngôn ngữ và nhất quán hơn trong các trường hợp biên.

#### 2. Lemmatization
Lemmatization đưa từ về **lemma** — dạng chuẩn tồn tại trong từ điển, có tính đến ngữ pháp (danh từ, động từ, tính từ). Kết quả luôn là một từ hợp lệ, nhưng tốc độ xử lý chậm hơn đáng kể so với stemming.

* **WordNet Lemmatizer**: Dựa trên cơ sở dữ liệu từ vựng tiếng Anh lớn là WordNet. Để hoạt động chính xác nhất, thuật toán này cần biết từ loại (Part-of-Speech - POS tag) của từ đó trong câu (VD: là danh từ, động từ, hay tính từ). Nếu không truyền POS tag, mặc định nó sẽ coi mọi từ là Danh từ (Noun).

#### 3. Chỉ số Collision Rate — Tỉ lệ đụng độ
- Collision Rate đo lường mức độ "hung hãn" của một phương pháp chuẩn hóa: tỉ lệ từ vựng bị gộp về cùng một dạng so với tổng từ vựng ban đầu.

$$\text{Collision Rate} = \frac{|V_{\text{before}}| - |V_{\text{after}}|}{|V_{\text{before}}|}$$

- **Ý nghĩa:** Tỉ lệ đụng độ càng cao nghĩa là không gian đặc trưng (feature space) càng được thu hẹp mạnh. Stemming thường có collision rate cao hơn Lemmatization.

In [ ]:
df['review_cleaned'] = df['review'].apply(basic_clean)
y = df['sentiment'].map({'positive': 1, 'negative': 0}).values
df[['review', 'review_cleaned', 'sentiment']].head()


In [ ]:
porter     = PorterStemmer()
snowball   = SnowballStemmer(language='english')
lemmatizer = WordNetLemmatizer()

all_words = set()
for text in df['review_cleaned']:
    all_words.update(str(text).split())
all_words = list(all_words)
total_unique_words = len(all_words)

def calculate_collision_rate(words, normalizer, is_lemmatizer=False):
    normalized_set = set()
    for word in words:
        if is_lemmatizer:
            normalized_set.add(normalizer.lemmatize(word))
        else:
            normalized_set.add(normalizer.stem(word))
    unique_normalized = len(normalized_set)
    collision_rate = (total_unique_words - unique_normalized) / total_unique_words
    return collision_rate, unique_normalized

porter_rate,   porter_vocab   = calculate_collision_rate(all_words, porter)
snowball_rate, snowball_vocab = calculate_collision_rate(all_words, snowball)
lemma_rate,    lemma_vocab    = calculate_collision_rate(all_words, lemmatizer, is_lemmatizer=True)

collision_df = pd.DataFrame({
    'Phương pháp':      ['Porter Stemmer', 'Snowball Stemmer', 'WordNet Lemmatizer'],
    'Vocab ban đầu':    [total_unique_words] * 3,
    'Vocab sau map':    [porter_vocab, snowball_vocab, lemma_vocab],
    'Collision Rate':   [f'{porter_rate:.2%}', f'{snowball_rate:.2%}', f'{lemma_rate:.2%}'],
})


def normalize_text(text, normalizer, is_lemmatizer=False):
    tokens = str(text).split()
    if is_lemmatizer:
        return ' '.join([normalizer.lemmatize(w) for w in tokens])
    return ' '.join([normalizer.stem(w) for w in tokens])

df['review_porter']   = df['review_cleaned'].apply(lambda x: normalize_text(x, porter))
df['review_snowball'] = df['review_cleaned'].apply(lambda x: normalize_text(x, snowball))
df['review_lemma']    = df['review_cleaned'].apply(lambda x: normalize_text(x, lemmatizer, is_lemmatizer=True))

def evaluate_model_with_metrics(text_series, y_target, method_name):
    tracemalloc.start()
    start_time = time.perf_counter()
    vectorizer = TfidfVectorizer(max_features=5000)
    X = vectorizer.fit_transform(text_series)
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(model, X, y_target, cv=5, scoring='accuracy', n_jobs=-1)
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    exec_time = time.perf_counter() - start_time
    return scores.mean(), exec_time, peak_mem / (1024 * 1024)

collision_df


In [ ]:
res_orig  = evaluate_model_with_metrics(df['review_cleaned'], y, 'Original Text')
res_port  = evaluate_model_with_metrics(df['review_porter'],  y, 'Porter Stemmer')
res_snow  = evaluate_model_with_metrics(df['review_snowball'],y, 'Snowball Stemmer')
res_lemma = evaluate_model_with_metrics(df['review_lemma'],   y, 'WordNet Lemmatization')

performance_df = pd.DataFrame([
    {'Phương pháp': 'Original Text',        'Accuracy': round(res_orig[0],  4), 'Time (s)': round(res_orig[1],  2), 'Peak RAM (MB)': round(res_orig[2],  2)},
    {'Phương pháp': 'Porter Stemmer',        'Accuracy': round(res_port[0],  4), 'Time (s)': round(res_port[1],  2), 'Peak RAM (MB)': round(res_port[2],  2)},
    {'Phương pháp': 'Snowball Stemmer',      'Accuracy': round(res_snow[0],  4), 'Time (s)': round(res_snow[1],  2), 'Peak RAM (MB)': round(res_snow[2],  2)},
    {'Phương pháp': 'WordNet Lemmatization', 'Accuracy': round(res_lemma[0], 4), 'Time (s)': round(res_lemma[1], 2), 'Peak RAM (MB)': round(res_lemma[2], 2)},
])

performance_df


### Phân tích kết quả

**1. Phân tích Mức độ Can thiệp và Tỉ lệ Đụng độ (Collision Rate)**

* **Snowball Stemmer (23.11%):** Là phương pháp cắt tỉa mạnh tay nhất, triệt tiêu hơn 54.000 từ vựng khác biệt (đưa tập từ vựng xuống chỉ còn 179,955 từ). Tỉ lệ đụng độ cực cao này cho thấy thuật toán đã gộp rất nhiều từ có chung gốc cấu tạo (dù ngữ cảnh hoặc từ loại có thể khác nhau) về làm một.
* **Porter Stemmer (14.79%):** ó mức độ can thiệp trung bình. Thuật toán này vẫn thu gọn đáng kể không gian đặc trưng nhưng ít "hung hãn" hơn so với bản nâng cấp Snowball của nó.
* **WordNet Lemmatizer (4.02%):** Có tỉ lệ đụng độ thấp nhất. Vì dựa trên từ điển và bảo toàn tính hợp lệ của từ (lemma), nó chỉ chuẩn hóa những từ thực sự có chung bổ đề gốc. Đây là phương pháp bảo toàn cấu trúc ngữ nghĩa an toàn nhất.

**2. So sánh hiệu năng phân loại**

Kết quả 5-fold cross-validation với Logistic Regression cho thấy:
* **Về Độ chính xác (Accuracy):** Việc giữ nguyên Original Text mang lại độ chính xác cao nhất (0.8874). Khi áp dụng chuẩn hóa, độ chính xác của cả 3 phương pháp đều giảm nhẹ (dao động từ 0.8864 đến 0.8869). Accuracy của bốn phương pháp chênh lệch rất nhỏ, phản ánh thực tế rằng với bộ dữ liệu đủ lớn và đặc trưng TF-IDF, cả stemming lẫn lemmatization đều không mang lại cải thiện đột phá.

* **Về Chi phí Tính toán (Time & RAM):**
    - **Original Text** tiêu tốn nhiều tài nguyên nhất: mất tới 72.75 giây và cần 196.32 MB RAM để huấn luyện.
    - **Porter Stemmer** tối ưu về tốc độ: cắt giảm gần một nửa thời gian huấn luyện (chỉ còn 37.08 giây) nhờ không gian từ vựng nhỏ gọn và thuật toán map đơn giản.
    - **Snowball Stemmer** tối ưu RAM (chỉ tốn 185.50 MB, giảm gần 11 MB so với gốc) nhờ có Collision Rate cao nhất (giảm chiều dữ liệu mạnh nhất). Thời gian chạy của Snowball (52.68s) và **WordNet Lemmatization** (53.61s) là tương đương nhau và đều tiết kiệm được ~20 giây so với dữ liệu gốc.

## 9. [Nâng cao] **Biểu diễn ngữ nghĩa bằng Sentence Transformer**: Sử dụng mô hình pretrained (ví dụ: all-MiniLM-L6-v2từ sentence-transformers).So sánh chất lượng phâncụm(K-Means,silhouettescore)vàhiệunăngphânloại(LinearSVM) giữa TF-IDF và Sentence Transformer embeddings. Giải thích sự khác biệt về mặt ngữ nghĩa.

### 9.1. Cơ sở lý thuyết về TF-IDF (Term Frequency - Inverse Document Frequency)

**TF-IDF** là một kỹ thuật thống kê trọng số truyền thống trong Xử lý Ngôn ngữ Tự nhiên (NLP) và Trích xuất Thông tin (Information Retrieval). Mục tiêu của TF-IDF là lượng hóa tầm quan trọng của một từ (term) đối với một văn bản (document) nằm trong một tập ngữ liệu (corpus) lớn. 

Khác với mô hình Bag-of-Words (BoW) cơ bản chỉ đếm tần suất xuất hiện, TF-IDF giải quyết bài toán "trọng số nhiễu": những từ xuất hiện nhiều trong một văn bản nhưng cũng xuất hiện ở hầu hết mọi văn bản khác (như *the, is, and* hoặc stopwords miền đặc thù) sẽ bị phạt và giảm trọng số.

Trọng số TF-IDF được cấu thành từ hai thành phần độc lập:

#### 9.1.1. Tần suất từ (Term Frequency - TF)
Đại lượng này đo lường mức độ xuất hiện của từ khóa $t$ trong văn bản $d$. Giả định cơ bản là từ nào xuất hiện càng nhiều trong một văn bản thì từ đó càng mang tính đại diện cho nội dung của văn bản đó. 

Cho $f_{t,d}$ là số lần từ $t$ xuất hiện trong văn bản $d$. Mặc dù có nhiều biến thể, công thức TF chuẩn hóa theo hàm logarit (Log-normalization) thường được sử dụng để giảm thiểu hiệu ứng bùng nổ trọng số của các văn bản quá dài:
$$TF(t, d) = 1 + \log(f_{t,d}) \quad \text{(nếu } f_{t,d} > 0 \text{, ngược lại } TF = 0)$$

#### 9.1.2. Nghịch đảo tần suất văn bản (Inverse Document Frequency - IDF)
Đại lượng này đo lường lượng thông tin (Information Content) mà từ $t$ mang lại trên toàn bộ tập ngữ liệu $D$. Nguyên lý cốt lõi là: những từ hiếm (xuất hiện ở ít văn bản) mang thông tin phân biệt cao hơn những từ phổ biến.

Gọi $N$ là tổng số văn bản trong tập ngữ liệu $D$ ($N = |D|$).
Gọi $df_t$ là số lượng văn bản chứa từ $t$ ($df_t = |\{d \in D : t \in d\}|$).
Chỉ số IDF được định nghĩa là logarit của nghịch đảo tỷ lệ văn bản chứa từ đó. Trong thực tiễn (như thư viện `scikit-learn`), công thức IDF thường được áp dụng cơ chế làm mịn (Smoothing) để tránh lỗi chia cho 0:
$$IDF(t, D) = \log \left( \frac{1 + N}{1 + df_t} \right) + 1$$

#### 9.1.3. Trọng số TF-IDF và Không gian Vector
Trọng số cuối cùng của từ $t$ trong văn bản $d$ thuộc tập $D$ là tích của hai đại lượng trên:
$$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$$

**Đặc điểm Không gian biểu diễn:**
Sau khi tính toán, mỗi văn bản $d$ sẽ được biểu diễn dưới dạng một vector trong không gian $\mathbb{R}^V$, với $V$ là kích thước từ điển (Vocabulary Size).
Đặc trưng của biểu diễn này là sinh ra một **Ma trận thưa (Sparse Matrix)** chiều không gian cực kỳ lớn. TF-IDF chỉ nắm bắt được **Sự trùng khớp từ vựng (Lexical Overlap)** mà hoàn toàn không có khả năng thấu hiểu ngữ cảnh hay từ đồng nghĩa.


In [ ]:
N_SAMPLE = 25000

samples, _ = train_test_split(
    df, 
    train_size=N_SAMPLE, 
    stratify=df['sentiment'], 
    random_state=42
)

samples = samples.reset_index(drop=True)

reviews = samples['review'].tolist()
labels = samples['sentiment'].map({'positive': 1, 'negative': 0}).tolist()

print(f"Tổng số mẫu dữ liệu: {len(samples)}")
print("Tỷ lệ phân phối lớp phân loại trong tập dữ liệu mẫu:")
print(samples['sentiment'].value_counts(normalize=True))


In [ ]:
# vectorizer = TfidfVectorizer(
#     max_features=10000,          
#     min_df=5,                    
#     max_df=0.95,                 
#     ngram_range=(1, 2)           
# )
# sparse_statistical_embeddings = vectorizer.fit_transform(reviews)
# print("Shape of sparse statisical embeddings", sparse_statistical_embeddings.shape)


In [ ]:
# sparse_path = r'..\data\processed\sparse'
# np.save(sparse_path, sparse_statistical_embeddings)


### 9.2. Cơ sở lý thuyết về Sentence Transformer (SBERT)

Để khắc phục rào cản ngữ nghĩa của TF-IDF, các kiến trúc Deep Learning dựa trên cơ chế Attention (đặc biệt là BERT) đã được áp dụng. **Sentence Transformer** (hay SBERT - Sentence-BERT) là một biến thể tối ưu hóa của kiến trúc BERT, được thiết kế chuyên biệt để trích xuất biểu diễn ngữ nghĩa của toàn bộ câu/văn bản dưới dạng các vector đặc (Dense Vectors) trong không gian có số chiều cố định.

#### 9.2.1. Kiến trúc Siamese Mạng nơ-ron (Siamese Network Architecture)
Mô hình ngôn ngữ BERT gốc yêu cầu đưa cả hai câu vào cùng một mạng (Cross-encoder) để tính toán sự tương đồng, dẫn đến chi phí tính toán bùng nổ khi áp dụng cho bài toán phân cụm (Clustering) trên dữ liệu lớn. 
Sentence Transformer giải quyết vấn đề này bằng cách áp dụng cấu trúc mạng Siamese (Bi-encoder). Các văn bản được đưa qua cùng một mô hình Transformer (ví dụ: MiniLM) một cách độc lập để tạo ra các embedding riêng biệt.

#### 9.2.2. Cơ chế Gom cụm (Pooling Strategy)
Khi một văn bản đi qua các lớp mã hóa (Encoder Layers) của Transformer, đầu ra là một chuỗi các vector biểu diễn cho từng từ (Token Embeddings): $H = [h_1, h_2, \dots, h_n]$, với $n$ là độ dài chuỗi token và $h_i \in \mathbb{R}^d$.

Để tổng hợp thông tin của toàn bộ câu thành một vector duy nhất $u \in \mathbb{R}^d$ (Sentence Embedding), mô hình SBERT sử dụng chiến lược **MEAN Pooling** (Lấy trung bình cộng không gian của tất cả các token embeddings):
$$u = \frac{1}{n} \sum_{i=1}^{n} h_i$$
*(Trong một số biến thể, mô hình có thể sử dụng biểu diễn của token đặc biệt `[CLS]` (CLS Pooling) hoặc `MAX Pooling`).*

<!-- #### 9.2.3. Khoảng cách Ngữ nghĩa và Độ tương đồng Cosine
Mục tiêu tối thượng của quá trình huấn luyện Sentence Transformer là đảm bảo: Các văn bản có ý nghĩa tương đồng sẽ nằm gần nhau trong không gian vector đa chiều, bất kể chúng có sử dụng chung từ vựng hay không.

Độ tương đồng giữa hai văn bản $A$ và $B$ (với vector biểu diễn tương ứng là $u_A, u_B$) được lượng hóa thông qua **Độ tương đồng Cosine (Cosine Similarity)**:
$$\text{Cosine}(A, B) = \cos(\theta) = \frac{u_A \cdot u_B}{\|u_A\| \|u_B\|} = \frac{\sum_{i=1}^{d} u_{A,i} u_{B,i}}{\sqrt{\sum_{i=1}^{d} u_{A,i}^2} \sqrt{\sum_{i=1}^{d} u_{B,i}^2}}$$ -->

**Đặc điểm Không gian biểu diễn:**
Sentence Transformer (như phiên bản `all-MiniLM-L6-v2`) tạo ra một **Ma trận đặc (Dense Matrix)** với số chiều rất nhỏ (thường là $d = 384$ hoặc $768$). Mô hình này giải quyết triệt để Curse of Dimensionality và sở hữu năng lực thấu hiểu **Sự tương đồng Ngữ nghĩa (Semantic Similarity)**, xử lý tốt các cấu trúc phủ định, ẩn dụ và từ đồng nghĩa.

In [ ]:
# transformer = SentenceTransformer('all-MiniLM-L6-v2') 
# dense_semantic_embeddings = transformer.encode(reviews, show_progress_bar=True)
# print("Shape of dense semantic embeddings", dense_semantic_embeddings.shape)
# # del transformer


In [ ]:
# dense_path = r'..\data\processed\dense'
# np.save(dense_path, dense_semantic_embeddings)


### 9.3. Tổng quan về Chỉ số Silhouette

**Chỉ số Silhouette (Silhouette Coefficient)** được giới thiệu lần đầu bởi Peter J. Rousseeuw vào năm 1987. Đây là một phương pháp định lượng thuộc nhóm **đánh giá nội tại (Internal Evaluation)** trong học máy không giám sát (Unsupervised Learning). Phương pháp này được sử dụng để đo lường chất lượng của các thuật toán phân cụm (Clustering) như K-Means, Hierarchical Clustering, khi tập dữ liệu không có nhãn thực tế (Ground Truth Labels).

Mục tiêu cốt lõi của phân cụm là tối đa hóa độ gắn kết giữa các phần tử trong cùng một cụm (Cohesion) và tối đa hóa độ tách biệt giữa các cụm khác nhau (Separation). Chỉ số Silhouette tích hợp cả hai tiêu chí này vào một biểu thức toán học duy nhất, cung cấp một thước đo trực quan và có giới hạn thống kê chặt chẽ để đánh giá mức độ phù hợp của từng điểm dữ liệu đối với cụm mà nó được gán.

#### 9.3.1. Cơ sở Toán học và Công thức 

Giả sử tập dữ liệu $D$ đã được phân thành $K$ cụm: $C = \{C_1, C_2, \dots, C_K\}$.
Xét một điểm dữ liệu (đối tượng) bất kỳ $i$ thuộc cụm $C_I$ (với $|C_I| > 1$). Quá trình tính toán hệ số Silhouette cho điểm $i$, ký hiệu là $s(i)$, được thực hiện qua ba bước:

##### Bước 1: Đo lường độ gắn kết nội cụm (Intra-cluster Cohesion) - Đại lượng $a(i)$
Đại lượng $a(i)$ được định nghĩa là khoảng cách trung bình từ điểm $i$ đến tất cả các điểm dữ liệu khác nằm trong *cùng cụm* $C_I$.
$$a(i) = \frac{1}{|C_I| - 1} \sum_{j \in C_I, i \neq j} d(i, j)$$
Trong đó, $d(i, j)$ là hàm khoảng cách (thông thường là khoảng cách Euclidean) giữa điểm $i$ và điểm $j$.
**Ý nghĩa:** $a(i)$ càng nhỏ chứng tỏ điểm $i$ càng nằm gần trung tâm cụm của nó, phản ánh mức độ gắn kết nội cụm càng cao.

##### Bước 2: Đo lường độ tách biệt liên cụm (Inter-cluster Separation) - Đại lượng $b(i)$
Đại lượng $b(i)$ là khoảng cách trung bình nhỏ nhất từ điểm $i$ đến mọi điểm thuộc một cụm $C_J$ bất kỳ khác với cụm $C_I$.
Đầu tiên, ta tính khoảng cách trung bình từ điểm $i$ đến tất cả các điểm thuộc cụm $C_J$:
$$d(i, C_J) = \frac{1}{|C_J|} \sum_{j \in C_J} d(i, j)$$
Sau đó, đại lượng $b(i)$ được xác định bằng giá trị cực tiểu của các khoảng cách này qua tất cả các cụm lân cận:
$$b(i) = \min_{J \neq I} d(i, C_J)$$
Cụm $C_J$ mang lại giá trị cực tiểu này được gọi là **"cụm lân cận gần nhất" (nearest neighboring cluster)** của điểm $i$.
**Ý nghĩa:** $b(i)$ càng lớn chứng tỏ điểm $i$ càng cách xa các cụm khác, phản ánh độ tách biệt càng tốt.

##### Bước 3: Tính toán Hệ số Silhouette cho đối tượng - $s(i)$
Hệ số Silhouette $s(i)$ là sự kết hợp của $a(i)$ và $b(i)$, được chuẩn hóa để giá trị luôn nằm trong một khoảng cố định:
$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$
*(Lưu ý: Nếu cụm $C_I$ chỉ có duy nhất 1 phần tử, ta quy ước $s(i) = 0$).*

Công thức trên có thể được viết lại dưới dạng hàm piecewise để dễ phân tích:
$$s(i) = 
\begin{cases} 
1 - \frac{a(i)}{b(i)}, & \text{nếu } a(i) < b(i) \\ 
0, & \text{nếu } a(i) = b(i) \\ 
\frac{b(i)}{a(i)} - 1, & \text{nếu } a(i) > b(i) 
\end{cases}$$

#### 9.3.2. Đánh giá và Diễn giải

Do mẫu số là $\max\{a(i), b(i)\}$, hệ số Silhouette luôn bị chặn trong đoạn $[-1, 1]$:
* **$s(i) \approx 1$:** Xảy ra khi $a(i) \ll b(i)$. Điều này khẳng định điểm $i$ được gán vào cụm cực kỳ chính xác (rất gần các điểm cùng cụm và rất xa cụm lân cận).
* **$s(i) \approx 0$:** Xảy ra khi $a(i) \approx b(i)$. Điểm $i$ đang nằm ở vùng không gian chồng lấn (boundary) giữa cụm hiện tại và cụm lân cận.
* **$s(i) < 0$:** Xảy ra khi $a(i) > b(i)$. Điểm $i$ nằm gần cụm lân cận hơn là cụm mà thuật toán đã gán cho nó. Đây là dấu hiệu của việc phân loại sai (Misclassification).

#### Chỉ số Silhouette Toàn cục (Global Silhouette Score)
Để đánh giá tổng thể chất lượng của mô hình với một số lượng cụm $K$ cụ thể, ta tính trung bình cộng hệ số $s(i)$ của toàn bộ $N$ điểm dữ liệu trong tập $D$:
$$S = \frac{1}{N} \sum_{i=1}^{N} s(i)$$
Giá trị $S$ toàn cục được sử dụng như một hàm mục tiêu (objective metric) để tối ưu hóa siêu tham số (Hyperparameter Tuning), điển hình là việc áp dụng **Phương pháp Elbow** hoặc trực tiếp tối đa hóa $S$ để tìm ra số cụm $K$ tối ưu nhất.

#### 9.3.3. Ưu điểm và Hạn chế 

* **Ưu điểm:** Khách quan, dễ diễn giải nhờ khoảng giá trị giới hạn $[-1, 1]$. Không phụ thuộc vào thuật toán phân cụm cụ thể.
* **Hạn chế:** 
  * *Calculation costs:* Độ phức tạp thuật toán là $O(N^2)$ để tính toán ma trận khoảng cách pairwise. Với các tập dữ liệu cực lớn, việc tính toán Silhouette Score đòi hỏi tài nguyên tính toán (RAM, CPU/GPU) khổng lồ hoặc phải sử dụng kỹ thuật lấy mẫu phân tầng (Stratified Sampling).
  * *Geometric Bias:* Silhouette sử dụng khoảng cách (thường là Euclidean), do đó nó có thiên hướng ưu ái các thuật toán tạo ra các cụm có hình cầu lồi (convex) và độ đặc cao (như K-Means). Đối với các thuật toán phân cụm dựa trên mật độ (như DBSCAN) hoặc dữ liệu có hình dáng đa diện phức tạp, Silhouette có thể trả về điểm số thấp ngay cả khi thuật toán hoạt động hoàn toàn chính xác.


In [ ]:
def process_pipeline(method, n_components=50, random_state=42):
    if method == "TF-IDF":
        return Pipeline([            
            (
                'dim_reduction',  
                TruncatedSVD(
                    n_components=n_components, 
                    random_state=random_state,
                    n_iter=10
                )
            ),
            
            (
                'norm', 
                Normalizer(norm='l2')
            )
        ])
        
    elif method == "SBERT":
        return Pipeline([
            (
                'dim_reduction', 
                PCA(
                    n_components=n_components,     
                    random_state=random_state,
                    svd_solver='full'      
                )
            ),
            
            (
                'norm', 
                Normalizer(norm='l2')
            )
        ])

def evaluate_clustering(X, name, k_list=[2, 5, 7]):
    results = []
    
    for k in k_list:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=50)
        clusters = kmeans.fit_predict(X)
        
        score = silhouette_score(X, clusters, metric='cosine')
        
        results.append({
            "Method": name,
            "k": k,
            "Silhouette Score": score
        })
    
    return pd.DataFrame(results)


Vì kết quả đã được tính toán và lưu trữ lần lượt ở 2 biến dense.npy và sparse.npy nên phần dưới sẽ load kết quả và analyze hiệu năng phân cụm và phân loại của hai phương pháp biểu diễn ngữ nghĩa này.

In [ ]:
X_sparse = np.load(r'..\data\processed\sparse.npy', allow_pickle=True).item()
pipeline_sparse = process_pipeline('TF-IDF', 20)
X_sparse = pipeline_sparse.fit_transform(X_sparse)
print("Shape of X_sparse", X_sparse.shape)

X_dense = np.load(r'..\data\processed\dense.npy')
pipeline_dense = process_pipeline('SBERT', 20)
X_dense = pipeline_dense.fit_transform(X_dense)
print("Shape of X_dense", X_dense.shape)



In [ ]:
result_silhouete_sparse = evaluate_clustering(X_sparse, "TF-IDF") 
result_silhouete_sparse


In [ ]:
result_silhouete_dense = evaluate_clustering(X_dense, "Sentence Transformer")
result_silhouete_dense


In [ ]:
result_clustering = pd.concat([result_silhouete_sparse, result_silhouete_dense], ignore_index=True)
result_clustering = result_clustering.pivot(index='k', columns='Method', values='Silhouette Score')

print("So sánh hiệu quả clustering")
print(result_clustering)


### Phân tích và Đánh giá Hiệu năng Phân cụm 

Bảng kết quả thực nghiệm trình bày chỉ số Silhouette Score nhằm đánh giá chất lượng phân cụm tự động trên hai không gian biểu diễn văn bản: mô hình ngôn ngữ học sâu (Sentence Transformer) và phương pháp thống kê truyền thống (TF-IDF), với các thiết lập số lượng cụm $k \in \{2, 5, 7\}$.

#### 1. Đánh giá Tổng quan về Chất lượng Phân cụm
Các giá trị Silhouette Score thu được dao động trong dải từ 0.12 đến 0.17. Trong bối cảnh phân tích dữ liệu dạng bảng (tabular data), đây có thể bị xem là mức độ phân tách yếu. Tuy nhiên, đối với lĩnh vực Xử lý Ngôn ngữ Tự nhiên (NLP), ngưỡng giá trị này là hoàn toàn điển hình và có giá trị tham khảo tốt. Nó phản ánh đặc thù cấu trúc của không gian dữ liệu văn bản: các văn bản thường mang tính đa nghĩa, đa chủ đề, dẫn đến hiện tượng chồng lấn ranh giới (overlapping boundaries) tự nhiên giữa các cụm. Tuy các cụm chưa đạt độ tách biệt hoàn hảo, nhưng cấu trúc tiềm ẩn (latent structure) của dữ liệu đã được các mô hình khai phá và bước đầu định hình.

#### 2. Phân tích sự vượt trội của TF-IDF trên thước đo Silhouette
Dữ liệu thực nghiệm chỉ ra một xu hướng nhất quán: TF-IDF vượt trội hơn Sentence Transformer trên toàn bộ các giá trị $k$. Cụ thể, tại $k=2$, hiệu năng của TF-IDF (0.169767) cao hơn SBERT (0.130285) xấp xỉ 30%. Hiện tượng này có thể được lý giải qua lăng kính toán học của độ đo Silhouette và bản chất biểu diễn đặc trưng của hai phương pháp:

* **Lexical Co-occurrence Bias:** TF-IDF kiến tạo một không gian vector đa chiều thưa thớt, định vị văn bản thuần túy dựa trên tần suất và sự đồng xuất hiện của các từ khóa cụ thể. Các văn bản chia sẻ một tệp từ vựng chuyên ngành chung sẽ hội tụ thành những điểm ảnh cực kỳ dày đặc (dense regions) trong không gian. Tính chất "vón cục" cục bộ này tối ưu hóa trực tiếp khoảng cách nội cụm (intra-cluster distance) – một thành tố cốt lõi giúp đẩy cao điểm số Silhouette.
* **Continuous Semantic Space:** Ngược lại, Sentence Transformer ánh xạ văn bản vào một không gian vector liên tục dựa trên ngữ cảnh và ý định (contextual semantics). Hai văn bản không dùng chung từ vựng nhưng đồng nghĩa vẫn được xếp gần nhau, tạo ra các cụm có độ trải rộng lớn hơn và đường bao phức tạp hơn. Thước đo Silhouette (vốn sử dụng khoảng cách Euclidean) mang tính thiên vị hình học, ưu ái các cấu trúc phân cụm hình cầu, cứng nhắc của TF-IDF và vô tình "trừng phạt" sự phân bố mềm dẻo, trải dài của SBERT.

#### 3. Đánh giá Cấu trúc Dữ liệu dựa trên tham số $k$
Quan sát sự dịch chuyển của điểm số theo tham số $k$, có thể rút ra kết luận về hình thái phân phối tự nhiên của bộ dữ liệu:
* Cả hai phương pháp biểu diễn đều đạt mức hiệu năng cực đại tại $k=2$. Sự hội tụ này cung cấp một tín hiệu thống kê mạnh mẽ: tập dữ liệu hiện tại sở hữu cấu trúc phân cực nhị phân rõ nét (ví dụ: sự phân ly rạch ròi giữa hai thái cực cảm xúc tích cực/tiêu cực, hoặc hai chủ đề lớn độc lập).
* Khi tăng độ phân giải của thuật toán lên $k=5$ và $k=7$, điểm Silhouette của cả hai phương pháp đều suy giảm. Sự sụt giảm hiệu năng này chỉ ra rằng việc ép buộc dữ liệu phân mảnh thành cấu trúc hạt mịn hơn (fine-grained clustering) đã làm suy yếu độ gắn kết nội bộ và gia tăng sự chồng chéo giữa các cụm lân cận.

#### 4. Tiểu kết
Dựa trên độ đo nội tại (internal metric) là Silhouette Score, phương pháp trích xuất đặc trưng TF-IDF đang cung cấp cấu trúc hình học tối ưu hơn cho thuật toán phân cụm. Tuy nhiên, tính chất của đo lường hình học không phản ánh toàn diện khả năng biểu diễn ngữ nghĩa. Trong khi TF-IDF xuất sắc trong việc gom nhóm các văn bản có cấu trúc từ vựng tương đồng, Sentence Transformer duy trì giá trị biểu diễn không gian ngữ cảnh liên tục.

### 9.4. Cơ sở lý thuyết về độ đo phụ thuộc ngưỡng (Threshold-dependent Metrics)

Trong học máy, mặc dù các mô hình phân loại thường xuất ra một điểm số liên tục hoặc xác suất (ví dụ: xác suất $60\%$ thuộc lớp Tích cực), quyết định phân lớp cuối cùng luôn đòi hỏi việc thiết lập một **ngưỡng phân định (Decision Threshold)** cụ thể (thường mặc định là $0.5$). Khi áp dụng ngưỡng này, các xác suất liên tục được chuyển hóa thành các nhãn rời rạc (Dương tính hoặc Âm tính). 

Để đánh giá hiệu năng thực tế của thuật toán sau khi đã đưa ra quyết định phân lớp cứng (Hard Classification) này, ta sử dụng hệ thống các **độ đo phụ thuộc ngưỡng (Threshold-dependent Metrics)**. Toàn bộ hệ thống này được xây dựng trên nền tảng của Ma trận Nhầm lẫn (Confusion Matrix) và Lý thuyết Xác suất Thống kê, giúp lượng hóa chi tiết từng dạng sai lầm mà mô hình mắc phải trên thực tế.

#### 9.4.1. Nền tảng Xác suất và Ma trận Nhầm lẫn (Confusion Matrix)

Giả sử không gian quan sát $X$ và không gian nhãn thực tế $Y \in \{0, 1\}$. Một bộ phân loại (Classifier) là một hàm ánh xạ $f: X \rightarrow \hat{Y}$, trong đó $\hat{Y} \in \{0, 1\}$ là nhãn dự đoán.

Đối với tập dữ liệu kiểm thử (Test set) có kích thước $N$, Ma trận nhầm lẫn phân rã tập hợp này thành 4 tập con rời rạc (Mutually exclusive subsets) dựa trên hàm chỉ thị $\mathbf{I}(\cdot)$:
* **True Positive (TP):** Lượng mẫu thỏa mãn $Y=1$ và $\hat{Y}=1$. $\quad TP = \sum_{i=1}^{N} \mathbf{I}(y_i = 1 \land \hat{y}_i = 1)$
* **False Positive (FP):** Lượng mẫu thỏa mãn $Y=0$ và $\hat{Y}=1$. $\quad FP = \sum_{i=1}^{N} \mathbf{I}(y_i = 0 \land \hat{y}_i = 1)$
* **True Negative (TN):** Lượng mẫu thỏa mãn $Y=0$ và $\hat{Y}=0$. $\quad TN = \sum_{i=1}^{N} \mathbf{I}(y_i = 0 \land \hat{y}_i = 0)$
* **False Negative (FN):** Lượng mẫu thỏa mãn $Y=1$ và $\hat{Y}=0$. $\quad FN = \sum_{i=1}^{N} \mathbf{I}(y_i = 1 \land \hat{y}_i = 0)$


#### 9.4.2. Hệ thống các Độ đo Cơ sở (Fundamental Metrics)

Các độ đo cơ sở là những đại lượng thống kê mô tả (Descriptive Statistics) phản ánh trực tiếp xác suất tiên nghiệm và xác suất hậu nghiệm của các dự đoán.

##### 9.4.2.1. Độ chính xác toàn cục (Accuracy)
* **Bản chất xác suất:** Accuracy ước lượng kỳ vọng toán học của việc mô hình đưa ra dự đoán trùng khớp với nhãn thực tế trên toàn bộ không gian mẫu.
  $$Accuracy = P(\hat{Y} = Y) = \frac{TP + TN}{N}$$
* **Hạn chế học thuật:** Accuracy nhạy cảm nghiêm trọng với phân phối tiên nghiệm của lớp (Class Prior Distribution). Nếu $P(Y=0) \gg P(Y=1)$ (dữ liệu mất cân bằng), giá trị Accuracy sẽ bị chi phối hoàn toàn bởi xác suất của lớp đa số, làm mất đi khả năng phản ánh sai số học máy đối với lớp thiểu số.

##### 9.4.2.2. Độ chuẩn xác (Precision / Positive Predictive Value - PPV)
* **Bản chất xác suất:** Precision là **xác suất có điều kiện** để một mẫu thực sự thuộc lớp Dương tính, biết rằng (given that) mô hình đã phân loại nó là Dương tính.
  $$Precision = P(Y=1 \mid \hat{Y}=1) = \frac{TP}{TP + FP}$$
* **Ý nghĩa:** Độ đo này lượng hóa mức độ tin cậy của các tín hiệu cảnh báo (Alarm Reliability). Precision suy giảm khi mô hình có xu hướng "báo động giả" (False Alarm / Type I Error).

##### 9.4.2.3. Độ nhạy (Recall / Sensitivity / True Positive Rate - TPR)
* **Bản chất xác suất:** Recall là xác suất có điều kiện để mô hình phân loại đúng một mẫu là Dương tính, biết rằng mẫu đó thực sự thuộc lớp Dương tính trong thực tế.
  $$Recall = P(\hat{Y}=1 \mid Y=1) = \frac{TP}{TP + FN}$$
* **Ý nghĩa:** Độ đo này lượng hóa Năng lực Khám phá (Discovery Power). Recall suy giảm khi mô hình có xu hướng "bỏ lọt" các đặc trưng quan trọng (Miss Rate / Type II Error).

##### 9.4.3. Sự đánh đổi Precision - Recall (Precision-Recall Trade-off)
Về mặt giải tích, khi ngưỡng phân định $\theta$ giảm dần, số lượng dự đoán $\hat{Y}=1$ tăng lên, kéo theo $TP$ và $FP$ cùng tăng. Sự gia tăng này làm tử số của Recall tăng (giúp Recall tiệm cận 1), nhưng đồng thời làm mẫu số của Precision tăng đột biến (khiến Precision suy giảm). Sự đánh đổi nghịch biến này là một định luật tất yếu trong lý thuyết học thống kê.



#### 9.4.4. Hệ thống các Độ đo Kết hợp (Combined Metrics)

Để đánh giá đồng thời nhiều khía cạnh mà không bị sai lệch bởi sự đánh đổi nói trên, các đại lượng toán học trung bình được áp dụng. Tuy nhiên, việc sử dụng Trung bình cộng (Arithmetic Mean) là sai lầm trong lý thuyết thông tin, vì nó cho phép bù trừ một độ đo rất thấp bằng một độ đo rất cao. Thay vào đó, **Trung bình điều hòa (Harmonic Mean)** được sử dụng.

##### 9.4.4.1. Điểm F tổng quát (F-measure / F-beta Score)
* **Cơ sở lý thuyết:** Điểm F là một trung bình điều hòa có trọng số giữa Precision và Recall. Khác với trung bình cộng, trung bình điều hòa sẽ "trừng phạt" (penalize) nghiêm khắc khi có sự chênh lệch lớn giữa Precision và Recall. Điểm F chỉ đạt giá trị cao khi cả hai đại lượng thành phần đều cao và xấp xỉ nhau.
* **Định nghĩa toán học:**
  $$F_{\beta} = (1 + \beta^2) \frac{Precision \times Recall}{(\beta^2 \times Precision) + Recall}$$
  Trong đó, $\beta$ là tham số điều khiển mức độ ưu tiên của Recall so với Precision. 
  * Dưới góc độ giải tích, $\frac{\partial F_{\beta}}{\partial Recall}$ sẽ lớn hơn $\frac{\partial F_{\beta}}{\partial Precision}$ khi $\beta > 1$, khiến hàm mục tiêu nhạy cảm hơn với sự sụt giảm của Recall.

##### 9.4.4.2. Điểm F1 (F1-Score)
* **Cơ sở lý thuyết:** Là trường hợp đặc biệt của $F_{\beta}$ khi $\beta = 1$. Ở trạng thái này, mô hình đặt trọng số và tầm quan trọng của Precision và Recall là hoàn toàn cân bằng nhau (Symmetric Weighting).
  $$F_1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$



#### 9.4.5. Phương pháp Tổng hợp Đa lớp (Multi-class Aggregation)

Khi giải quyết bài toán phân loại đa lớp (Multi-class Classification) với tập hợp nhãn $C = \{c_1, c_2, \dots, c_k\}$, các độ đo cơ sở và kết hợp (như F1-Score) phải được tính toán cho toàn bộ mô hình thông qua ba phương pháp tổng hợp (Aggregation Methods) có bản chất thống kê khác biệt:

##### 9.4.5.1. Tổng hợp Vĩ mô (Macro-Average)
* **Cơ sở lý thuyết:** Đánh giá độ đo (Metric) trên từng lớp $i$ một cách độc lập, sau đó lấy trung bình cộng không trọng số của tất cả các lớp.
  $$Metric_{macro} = \frac{1}{|C|} \sum_{i=1}^{|C|} Metric_i$$
* **Ý nghĩa:** Phương pháp này coi mọi lớp có tầm quan trọng như nhau, bất chấp kích thước (Support) của chúng. Nó là tiêu chuẩn vàng để phát hiện mô hình bị chệch (Bias) về phía lớp đa số, vì một lớp thiểu số bị đoán sai sẽ kéo tụt mạnh điểm số tổng thể.

##### 9.4.5.2. Tổng hợp Vi mô (Micro-Average)
* **Cơ sở lý thuyết:** Gộp tổng các giá trị $TP, FP, FN$ của tất cả các lớp lại với nhau trước, sau đó mới tính toán Metric một lần duy nhất từ tổng các giá trị đó.
  $$Precision_{micro} = \frac{\sum_{i=1}^{|C|} TP_i}{\sum_{i=1}^{|C|} TP_i + \sum_{i=1}^{|C|} FP_i}$$
* **Ý nghĩa:** Trong môi trường phân loại đa lớp đơn nhãn (Multi-class Single-label), $Micro-F1$, $Micro-Precision$ và $Micro-Recall$ luôn bằng nhau và chính xác bằng *Accuracy*. Phương pháp này bị chi phối hoàn toàn bởi hiệu suất dự đoán trên lớp chiếm đa số.

##### 9.4.5.3. Tổng hợp có Trọng số (Weighted-Average)
* **Cơ sở lý thuyết:** Tính toán Metric cho từng lớp độc lập, sau đó lấy trung bình cộng có nhân với trọng số là tỷ lệ phân phối tiên nghiệm của lớp đó (Tỷ lệ mẫu của lớp $i$ trên tổng số mẫu $N$).
  $$Metric_{weighted} = \sum_{i=1}^{|C|} \left( \frac{N_i}{N} \times Metric_i \right)$$
* **Ý nghĩa:** Là giải pháp dung hòa khi dữ liệu bị mất cân bằng nhưng ta vẫn muốn thành tích trên các lớp đa số đóng góp nhiều hơn vào điểm số cuối cùng. Tuy nhiên, nó thiếu đi khả năng cảnh báo lỗi nghiêm trọng trên các lớp thiểu số như Macro-Average.

### 9.5. Cơ sở lý thuyết về các độ đo không phụ thuộc ngưỡng: Đường cong ROC và Chỉ số AUC

Theo phân tích ở phần trước, hệ thống các độ đo cơ sở (như Accuracy, Precision, F1-Score) cung cấp góc nhìn chi tiết về hiệu năng mô hình, nhưng chúng mang một hạn chế về mặt lý thuyết: **chúng phụ thuộc hoàn toàn vào giá trị của ngưỡng phân định $\theta$**. Nếu thay đổi ngưỡng, các chỉ số này sẽ lập tức biến động. Do đó, việc đánh giá tại một ngưỡng cố định không phản ánh được bức tranh toàn cảnh về sức mạnh nội tại của thuật toán.

Để khắc phục rào cản này, đồng thời để so sánh khách quan năng lực phân tách cốt lõi giữa các mô hình khác nhau trên **mọi ngưỡng có thể có**, ta cần sử dụng hệ thống các **độ đo không phụ thuộc ngưỡng (Threshold-invariant Metrics)**. Tiêu biểu và mang tính nền tảng nhất trong nhóm này là Đường cong Đặc trưng Hoạt động của Bộ thu (ROC Curve) và Chỉ số Diện tích dưới đường cong (AUC).

#### 9.5.1. Đường cong Đặc trưng Hoạt động của Bộ thu (ROC Curve)

##### 9.5.1.1. Nguồn gốc và Khái niệm
Đường cong ROC (Receiver Operating Characteristic) bắt nguồn từ Lý thuyết Phát hiện Tín hiệu (Signal Detection Theory) trong radar quân sự thời Thế chiến II, sau đó được ứng dụng rộng rãi trong y học và học máy. ROC là một đồ thị hai chiều minh họa hiệu năng của một mô hình phân loại nhị phân khi ngưỡng phân định $\theta$ thay đổi liên tục từ 1.0 giảm dần về 0.0.

##### 9.5.1.2. Không gian ROC (ROC Space)
Không gian ROC được định nghĩa trên hệ trục tọa độ Đề-các (Cartesian coordinate system) với miền giá trị $[0, 1] \times [0, 1]$:
* **Trục hoành (X-axis):** Tỷ lệ Dương tính giả (False Positive Rate - $FPR$), còn gọi là $1 - \text{Specificity}$. Nó đo lường tỷ lệ các mẫu thực tế là Âm tính nhưng bị dự đoán nhầm thành Dương tính.
  $$FPR(\theta) = \frac{FP(\theta)}{FP(\theta) + TN(\theta)}$$
* **Trục tung (Y-axis):** Tỷ lệ Dương tính thật (True Positive Rate - $TPR$), còn gọi là Độ nhạy (Sensitivity) hay Recall. Nó đo lường tỷ lệ các mẫu thực tế là Dương tính và được nhận diện chính xác.
  $$TPR(\theta) = \frac{TP(\theta)}{TP(\theta) + FN(\theta)}$$

##### 9.5.1.3. Hình thái đường cong và Điểm cực trị
Một đường cong ROC được tạo ra bằng cách đánh giá cặp tọa độ $(FPR(\theta), TPR(\theta))$ tại mọi giá trị có thể có của $\theta$. Quá trình này tạo ra một đường cong tiệm cận từ tọa độ $(0,0)$ đến $(1,1)$.
* **Điểm $(0, 0)$:** Tương ứng với $\theta = \infty$ (hoặc $\theta = 1.0$ với xác suất). Mô hình dự đoán mọi mẫu đều là Âm tính. Không có sai lầm FP, nhưng cũng không tìm được TP nào.
* **Điểm $(1, 1)$:** Tương ứng với $\theta = -\infty$ (hoặc $\theta = 0.0$). Mô hình dự đoán mọi mẫu đều là Dương tính. Tìm được toàn bộ TP, nhưng đồng thời gán sai toàn bộ TN thành FP.
* **Điểm $(0, 1)$:** Điểm lý tưởng (Perfect Classification). Mô hình phân tách hoàn hảo hai lớp mà không mắc bất kỳ sai lầm nào. Đường ROC càng uốn cong về phía điểm $(0, 1)$ thì mô hình càng mạnh.
* **Đường chéo $y = x$:** Tương ứng với một bộ phân loại đoán mò ngẫu nhiên (Random Classifier). Bất kỳ mô hình nào có đường ROC nằm dưới đường chéo này đều cho kết quả tệ hơn cả tung đồng xu.


#### 9.5.2. Chỉ số Diện tích dưới Đường cong (Area Under the Curve - AUC)

Đường cong ROC cung cấp một cái nhìn trực quan, nhưng để so sánh định lượng giữa nhiều mô hình, ta cần một giá trị vô hướng (Scalar). **AUC-ROC** (hay gọi tắt là AUC) chính là diện tích không gian hai chiều nằm dưới đường cong ROC.

##### 9.5.2.1. Định nghĩa Tích phân (Integral Definition)
Về mặt giải tích, AUC là tích phân xác định của hàm số $TPR$ theo biến $FPR$:
$$AUC = \int_{0}^{1} TPR(FPR) \, d(FPR)$$
Do miền giá trị của cả FPR và TPR đều nằm trong khoảng $[0, 1]$, diện tích tối đa của không gian ROC là $1 \times 1 = 1$. Do đó, $AUC \in [0, 1]$.

##### 9.5.2.2. Ý nghĩa Xác suất và Thống kê (Probabilistic Interpretation)
Đây là khía cạnh quan trọng nhất của AUC trong các nghiên cứu hàn lâm. Về mặt xác suất toán học, AUC tương đương với **Xác suất để bộ phân loại gán một điểm số (score) cho một mẫu Dương tính lấy ngẫu nhiên cao hơn một mẫu Âm tính lấy ngẫu nhiên.**

Gọi $X^+$ là một biến ngẫu nhiên đại diện cho tập hợp các mẫu Dương tính, $X^-$ là tập hợp các mẫu Âm tính. Gọi $f(x)$ là hàm cho điểm (Scoring Function) của mô hình (ví dụ: xác suất $P(Y=1|x)$).
$$AUC = P(f(X^+) > f(X^-))$$

**Mối liên hệ với Thống kê phi tham số:**
Hệ thức xác suất trên cho thấy AUC có sự tương đương toán học trực tiếp với **Kiểm định Wilcoxon-Mann-Whitney (Mann-Whitney U Test)**. 
Cho một tập dữ liệu gồm $n_1$ mẫu Dương tính và $n_2$ mẫu Âm tính, AUC có thể được ước lượng rời rạc bằng công thức:
$$AUC = \frac{1}{n_1 n_2} \sum_{i=1}^{n_1} \sum_{j=1}^{n_2} \mathbf{I}(f(x_i^+) > f(x_j^-))$$
Trong đó, $\mathbf{I}(\cdot)$ là hàm chỉ thị (Indicator Function), nhận giá trị $1$ nếu điều kiện đúng, và $0$ nếu sai (nếu hai điểm số bằng nhau, ta lấy giá trị $0.5$).


#### 9.5.3. Ưu điểm và Đặc tính Thống kê của AUC-ROC

Việc sử dụng AUC-ROC làm hàm mục tiêu đánh giá mang lại hai đặc tính bất biến cực kỳ quan trọng:

##### 9.5.3.1. Tính bất biến theo tỷ lệ (Scale-Invariance)
AUC đo lường khả năng **xếp hạng (Ranking)** của các điểm dự đoán hơn là giá trị tuyệt đối của chúng. Dù hàm dự đoán $f(x)$ trả về xác suất từ $0$ đến $1$, hay điểm số vô hướng từ $-100$ đến $+100$ (như khoảng cách đến siêu phẳng trong SVM), chỉ cần thứ tự sắp xếp của các điểm dữ liệu không thay đổi, giá trị AUC sẽ được bảo toàn.

##### 9.5.3.2. Tính bất biến theo ngưỡng (Threshold-Invariance)
Vì AUC tích phân hóa qua tất cả các ngưỡng $\theta \in [0, 1]$, nó đánh giá sức mạnh phân tách nội tại của mô hình một cách toàn cục. Điều này đặc biệt hữu ích trong giai đoạn huấn luyện và so sánh thuật toán, trước khi ta quyết định chọn một ngưỡng tối ưu cho ứng dụng thực tiễn dựa trên chi phí rủi ro nghiệp vụ (Business Cost Function).

##### 9.5.3.3. Độ bền vững với Mất cân bằng dữ liệu (Robustness to Imbalance)
Do $TPR$ và $FPR$ đều được tính toán trên tử số và mẫu số của cùng một lớp (TPR chỉ phụ thuộc tập Dương tính, FPR chỉ phụ thuộc tập Âm tính), đường ROC không bị dịch chuyển khi tỷ lệ phân phối tiên nghiệm (Prior Class Distribution) thay đổi. Nhờ đó, AUC vẫn là một độ đo khách quan ngay cả khi dữ liệu bị mất cân bằng (Imbalanced Data). 

*(Lưu ý: Mặc dù ROC-AUC không bị bóp méo bởi mất cân bằng lớp, trong các trường hợp dữ liệu lệch cực đoan (Ví dụ: 1:1000), số lượng FP nhỏ có thể bị chìm lấp trong tập Âm tính khổng lồ, khiến FPR duy trì ở mức thấp giả tạo. Trong các kịch bản này, Đường cong Precision-Recall (PR-AUC) thường được sử dụng như một chỉ số bổ trợ chuyên sâu hơn).*

In [ ]:
labels = samples['sentiment'].map({'positive': 1, 'negative': 0}).tolist()


In [ ]:
X_dense = np.load(r'..\data\processed\dense.npy')
X_sparse = np.load(r'..\data\processed\sparse.npy', allow_pickle=True).item()


In [ ]:
y = labels
X_train_sparse, X_test_sparse, y_train_sparse, y_test_sparse = train_test_split(X_sparse, labels, test_size=0.2, random_state=42)
X_train_dense, X_test_dense, y_train_dense, y_test_dense = train_test_split(X_dense, labels, test_size=0.2, random_state=42)

def get_threshold_dependent_metrics(y_true, y_pred, model_name):
    """
    Tính toán các độ đo phụ thuộc vào một ngưỡng phân định cụ thể (Accuracy, Precision, Recall, F1).
    Yêu cầu đầu vào y_pred phải là nhãn rời rạc (0 hoặc 1).
    """
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    
    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision (Macro)': precision,
        'Recall (Macro)': recall,
        'F1-Score (Macro)': f1
    }

def get_threshold_independent_metrics(y_true, y_scores, model_name):
    """
    Tính toán các độ đo đánh giá toàn cục trên mọi ngưỡng (ROC, AUC).
    Yêu cầu đầu vào y_scores phải là giá trị liên tục (xác suất hoặc distance to hyperplane).
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    return {
        'Model': model_name,
        'FPR': fpr,
        'TPR': tpr,
        'AUC': roc_auc,
        'Thresholds': thresholds 
    }
    

def evaluate_classification(X_train, X_test, y_train, y_test, model_name):
    svm = LinearSVC(random_state=42, max_iter=2000, dual=False)
    svm.fit(X_train, y_train)
    
    y_pred = svm.predict(X_test) 
    
    y_scores = svm.decision_function(X_test) 
    
    metrics = get_threshold_dependent_metrics(y_test, y_pred, model_name)
    ROC = get_threshold_independent_metrics(y_test, y_scores, model_name)
    
    return metrics, ROC


In [ ]:
result_metrics = []
result_ROC = []


metrics_tfidf, roc_tfidf = evaluate_classification(
    X_train_sparse, X_test_sparse, y_train_sparse, y_test_sparse, "TF-IDF"
)
result_metrics.append(metrics_tfidf)
result_ROC.append(roc_tfidf)


metrics_st, roc_st = evaluate_classification(
    X_train_dense, X_test_dense, y_train_dense, y_test_dense, "Sentence Transformer"
)
result_metrics.append(metrics_st)
result_ROC.append(roc_st)

display(pd.DataFrame(result_metrics))


In [ ]:
def plot_roc(ROC):
    plt.figure(figsize=(8, 6))
    
    colors = ['#2980b9', '#e74c3c'] 
    
    for i, data in enumerate(ROC):
        plt.plot(data['FPR'], data['TPR'], color=colors[i], lw=2, 
                 label=f"{data['Model']} (AUC = {data['AUC']:.4f})")
        
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Guess')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)', fontsize=12)
    plt.ylabel('True Positive Rate (TPR)', fontsize=12)
    plt.title('Biểu đồ đường: So sánh ROC giữa TF-IDF và Sentence Transformer', fontsize=16, fontweight='bold', pad=15)
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_roc(result_ROC)


In [ ]:
def plot_metrics(metrics):
    df_melted = metrics.melt(id_vars='Model', var_name='Metric', value_name='Score')
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_melted, x='Metric', y='Score', hue='Model', palette='Set2')
    
    plt.ylim([0.5, 1.0]) 
    plt.title('Biểu đồ cột: So sánh Metrics giữa TF-IDF và Sentence Transformer', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Chỉ số (Metric)', fontsize=12)
    plt.ylabel('Điểm số (Score)', fontsize=12)
    plt.legend(loc='lower right', title='Phương pháp Biểu diễn')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_metrics(pd.DataFrame(result_metrics))


### Phân tích và Đánh giá Hiệu năng Phân loại 
Phần này trình bày các đánh giá định lượng về khả năng phân loại của hai phương pháp biểu diễn văn bản: TF-IDF và Sentence Transformer (SBERT), sử dụng thuật toán Support Vector Machine với nhân tuyến tính (LinearSVC). Đánh giá được thực hiện thông qua các chỉ số phân loại cục bộ và khả năng phân định ngưỡng qua đường cong ROC.

#### 1. Đánh giá dựa trên các chỉ số cục bộ (Accuracy, Precision, Recall, F1-Score)
Dữ liệu từ bảng tổng hợp và biểu đồ cột cho thấy sự vượt trội toàn diện của phương pháp TF-IDF so với Sentence Transformer trên hệ thống phân loại tuyến tính.

* **Hiệu năng của TF-IDF:** Mô hình đạt mức điểm xuất sắc với Accuracy và F1-Score (Macro) hội tụ ở mức **0.8954** (~89.5%). Các chỉ số Precision và Recall cân bằng hoàn hảo, chứng tỏ siêu phẳng (hyperplane) do LinearSVC thiết lập đã chia cắt không gian dữ liệu một cách cực kỳ chính xác và không bị thiên lệch.
* **Hiệu năng của Sentence Transformer (SBERT):** Tương tự, độ phân loại của SBERT duy trì tính cân bằng ổn định giữa các nhãn, nhưng điểm số tổng thể chỉ đạt **0.8234** (thấp hơn khoảng 7.2% so với TF-IDF). 

#### 2. Phân tích Khả năng Phân định qua Đường cong ROC và AUC
Đường cong ROC cung cấp góc nhìn sâu sắc về độ tự tin của mô hình thông qua hàm quyết định khoảng cách (`decision_function` của LinearSVC).

* **Sự thống trị của TF-IDF:** Đường cong ROC của TF-IDF hoàn toàn bao trùm lên SBERT với chỉ số **AUC đạt 0.8764**. Điều này chứng minh rằng khoảng cách từ các điểm dữ liệu TF-IDF đến ranh giới quyết định (decision boundary) là rất rõ ràng. Mô hình cực kỳ tự tin khi phân loại và duy trì được Tỷ lệ Dương tính Thật (TPR) cao ngay cả khi siết chặt ngưỡng rủi ro.
* **Hạn chế của SBERT:** Với **AUC đạt 0.8086**, đường cong của SBERT tiệm cận đường phân phối ngẫu nhiên sớm hơn. Các điểm dữ liệu SBERT có xu hướng nằm quá sát siêu phẳng tuyến tính, khiến mô hình gặp khó khăn trong việc gán xác suất tự tin rõ rệt cho các mẫu dữ liệu ở vùng biên.

#### 3. Phân tích chuyên sâu: Hiện tượng "Nút thắt Tuyến tính" 
Sự vượt trội của TF-IDF trước SBERT trong thực nghiệm này không chứng minh TF-IDF "thông minh" hơn, mà nó phản ánh sự tương thích toán học giữa không gian biểu diễn và bộ phân loại LinearSVC:

* **Sự tương thích hoàn hảo (TF-IDF + LinearSVC):** Ma trận TF-IDF tạo ra một không gian vector siêu đa chiều nhưng cực kỳ thưa (sparse, high-dimensional space). Trong lý thuyết học máy, dữ liệu càng thưa và đa chiều thì khả năng phân tách tuyến tính (linearly separable) càng cao. LinearSVC đóng vai trò như một bộ gán trọng số trực tiếp cho các "từ khóa phân cực" (polar keywords). Nếu văn bản xuất hiện từ khóa mang tính quyết định, nó lập tức bị đẩy về một phía của siêu phẳng.
* **Nút thắt tuyến tính (SBERT + LinearSVC):** Sentence Transformer ánh xạ văn bản vào một không gian vector dày đặc (dense vectors - ví dụ 384 chiều). Trong không gian này, ngữ nghĩa được biểu diễn liên tục; hai văn bản mang cảm xúc đối lập ("rất ngon" và "rất dở") có thể nằm rất gần nhau vì cùng chia sẻ ngữ cảnh "đánh giá ẩm thực". Cấu trúc phân phối này mang tính **phi tuyến (non-linear)** rất cao. Khi áp dụng LinearSVC – một thuật toán chỉ có khả năng vẽ các mặt phẳng cắt thẳng tắp – nó không đủ độ uốn dẻo để len lỏi và bóc tách các vùng ngữ nghĩa phức tạp này. Do đó, điểm số của SBERT bị giới hạn bởi chính bộ phân loại tuyến tính.

#### 4. Tiểu kết
Thực nghiệm cho thấy bộ phân loại tuyến tính (LinearSVC) khai thác tối đa sức mạnh của không gian đặc trưng từ vựng phân tán (TF-IDF), mang lại hiệu năng xấp xỉ 90%. Trái lại, biểu diễn ngữ nghĩa dày đặc của Sentence Transformer gặp hiện tượng "nút thắt tuyến tính", khiến tiềm năng của mô hình bị kìm hãm. Kết quả này củng cố nguyên lý thiết kế pipeline trong NLP: Biểu diễn thưa (Sparse) phù hợp với các bộ phân loại tuyến tính cơ bản, trong khi biểu diễn dày đặc (Dense) yêu cầu các bộ phân loại phi tuyến tính phức tạp (như Multi-layer Perceptron hoặc SVM với nhân RBF) để tối ưu hóa hiệu năng phân loại cảm xúc.


## 10. Tổng kết

Xuyên suốt notebook này, chúng ta đã tiến hành xây dựng, phân tích và đánh giá định lượng một cách hệ thống toàn bộ vòng đời tiền xử lý văn bản (Text Preprocessing Pipeline) và các chiến lược biểu diễn đặc trưng (Feature Representation) trong lĩnh vực Xử lý Ngôn ngữ Tự nhiên (NLP). 

Dựa trên các kết quả thực nghiệm với bài toán Phân tích Cảm xúc (Sentiment Analysis) trên tập dữ liệu IMDB, có thể rút ra những kết luận mang tính học thuật và thực tiễn như sau:

**1. Vai trò của Tiền xử lý và Làm sạch dữ liệu (Data Cleaning & Normalization)**
* Việc loại bỏ nhiễu (thẻ HTML, ký tự đặc biệt, URL) là tối quan trọng, giúp nén triệt để chiều dài văn bản và làm sạch không gian từ vựng. Tuy nhiên, các thao tác can thiệp sâu như loại bỏ Stop Words hay Stemming/Lemmatization cần được thực hiện một cách cẩn trọng. 
* Thực nghiệm chỉ ra rằng, đối với bài toán phân tích cảm xúc, việc áp dụng danh sách Stop Words mặc định có thể vô tình triệt tiêu các "từ phủ định" mang ý nghĩa quyết định, làm giảm lượng thông tin phân loại (Mutual Information). Tương tự, phương pháp WordNet Lemmatization cho thấy sự an toàn về mặt bảo toàn ngữ nghĩa so với các thuật toán cắt tỉa "hung hãn" (high collision rate) như Snowball Stemmer, giúp cân bằng giữa chi phí tính toán và tính toàn vẹn của dữ liệu.

**2. Chiến lược Tokenization (Tokenization Strategies)**
* Việc lựa chọn Tokenizer quyết định trực tiếp đến hình thái của không gian đặc trưng. Phân rã mức từ (Word-level) là nền tảng vững chắc cho các mô hình thống kê (BoW, TF-IDF). Trong khi đó, phân rã mức từ phụ (Subword BPE) chứng minh được sức mạnh vượt trội trong việc kiểm soát kích thước từ điển và triệt tiêu hoàn toàn tỷ lệ từ chưa biết (OOV rate), trở thành tiêu chuẩn vàng cho các mô hình ngôn ngữ lớn (Transformers).

**3. Nghịch lý giữa Biểu diễn Từ vựng (Lexical) và Ngữ nghĩa (Semantic)**
Thực nghiệm chuyên sâu so sánh giữa TF-IDF và Sentence Transformer (SBERT) đã làm sáng tỏ một đặc tính quan trọng trong thiết kế hệ thống học máy:
* **Sức mạnh của ma trận thưa:** TF-IDF kết hợp cùng LinearSVC tạo ra một hệ thống cực kỳ mạnh mẽ cho phân loại cảm xúc (Accuracy ~89.5%). Việc bảo toàn không gian từ vựng nguyên bản giúp bộ phân loại tuyến tính dễ dàng gán trọng số cho các từ khóa phân cực rõ rệt.
* **Nút thắt phân loại và Thiên vị hình học:** Mô hình học sâu SBERT bị "trừng phạt" trên thang đo Silhouette do tính chất phân bố ngữ nghĩa liên tục thay vì co cụm cục bộ. Đồng thời, khi kết hợp SBERT với LinearSVC, mô hình gặp phải hiện tượng "nút thắt tuyến tính" (linear bottleneck) do một siêu phẳng thẳng không thể bóc tách hiệu quả không gian ngữ nghĩa nén phi tuyến tính, dẫn đến hiệu năng (Accuracy ~82.3%) thấp hơn so với phương pháp truyền thống.

**Lời kết:**
Không có một "đáp án vạn năng" (one-size-fits-all) trong Xử lý Ngôn ngữ Tự nhiên. Việc xây dựng pipeline NLP đòi hỏi sự thấu hiểu sâu sắc về bản chất của tập dữ liệu (phụ thuộc vào từ khóa hay ngữ cảnh) và sự **tương thích toán học** giữa phương pháp trích xuất đặc trưng và bộ máy phân loại. Biểu diễn thưa (Sparse/TF-IDF) tỏa sáng với các mô hình tuyến tính đơn giản, trong khi biểu diễn đặc (Dense/SBERT) đòi hỏi các thuật toán phi tuyến tính phức tạp (như Mạng nơ-ron) để thực sự giải phóng toàn bộ tiềm năng thấu hiểu ngôn ngữ.